# IntrinsicZernikes v3 — plots straight from the certified Butler collection

Reads the **certified** `intrinsicZernikes` calibration from
`u/gmegias/calib/DM-55048/intrinsicZernikes.v3` in `/repo/main` and plots it — no
parquet files: this is exactly the object a consumer pipeline
(`ip_isr.IntrinsicZernikes`) would `butler.get`.

The collection is a `CALIBRATION` collection with 205 detectors × 6 physical
filters = 1230 datasets (certified with an unbounded timespan).

Each per-(detector, filter) calib carries two field maps:
* **`field_x_ocs` / `values_ocs`** — the telescope-fixed (OCS) contribution, shared
  across detectors, on a full disk out to the **donut-data edge (~1.73°)**. There is
  no data past that (donuts stop at ~1.72°), so the disk does not reach the outer
  detector corners (~2.05°) — that is a data limit, not a plotting artifact, and the
  frames below are set to the data radius so the disk fills them.
* **`field_x` / `values`** — the camera-fixed (CCS) contribution.

**What's new in v3 (vs v2):** the CCS field is now sampled on a per-detector
**footprint grid** (~49 points across each CCD), with the CCD height sampled at
**every point** rather than collapsed to a single Z4 piston at the CCD centre. This
preserves the intra-CCD height structure from the height map (median ~0.17 µm,
up to ~0.5 µm peak-to-peak in Z4) that v2 flattened away. A CCD's CCS map is
therefore only defined **over its own footprint** — exactly where a consumer queries
it. The ~16 field-edge CCDs whose footprints fall past the data support keep the v2
whole-focal-plane field.

`getIntrinsicZernikes(field_x, field_y, rotTelPos)` returns the sum (CCS as-is +
OCS rotated by `rotTelPos`) — the actual intrinsic aberration a consumer sees.

**Env:** run with the LSST stack sourced (`source /sdf/group/rubin/sw/d_latest/loadLSST.bash && setup lsst_distrib -t d_latest`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler

REPO = "/repo/main"
COLL = "LSSTCam/calib/DM-55333/intrinsicZernikes.v2/"
INSTRUMENT = "LSSTCam"
FP_RADIUS_DEG = 1.75  # science field-of-view radius for the plot frame

butler = Butler(REPO)
info = butler.collections.get_info(COLL)
refs = list(butler.registry.queryDatasets("intrinsicZernikes", collections=COLL, findFirst=True))
DETECTORS = sorted({r.dataId["detector"] for r in refs})
FILTERS = sorted({r.dataId["physical_filter"] for r in refs})
print(f"collection : {info.name}  (type={info.type!s})")
print(f"datasets   : {len(refs)}   ({len(DETECTORS)} detectors × {len(FILTERS)} filters)")
print(f"filters    : {FILTERS}")
print(f"detectors  : {DETECTORS[0]}..{DETECTORS[-1]}")

In [ ]:
def load_calib(detector, physical_filter):
    """Fetch one certified IntrinsicZernikes calib from the Butler."""
    return butler.get(
        "intrinsicZernikes", collections=COLL,
        instrument=INSTRUMENT, detector=detector, physical_filter=physical_filter,
    )

# --- pick a filter, then load EVERY detector's calib once --------------------
FILT = "i_39"       # any value in FILTERS
# ---------------------------------------------------------------------------
print(f"loading {len(DETECTORS)} detector calibs for filter {FILT} ... (~1-2 min)")
calibs = {d: load_calib(d, FILT) for d in DETECTORS}
noll = list(np.asarray(calibs[DETECTORS[0]].noll_indices))

# OCS: identical across detectors (shared telescope field, full science disk)
oc = calibs[DETECTORS[0]]
ocs_x = np.asarray(oc.field_x_ocs, float)
ocs_y = np.asarray(oc.field_y_ocs, float)
ocs_v = np.asarray(oc.values_ocs, float)                 # (n_ocs, n_noll)

# plot frame = the actual data radius (the maps stop at the donut-data edge,
# ~1.73°; there is no data past it), so the disk fills the frame instead of
# looking cut off inside the 1.75° science circle.
DATA_R = float(np.hypot(ocs_x, ocs_y).max())

# CCS: each detector carries its OWN samples -> assemble the whole camera.
# Footprint CCDs (~49 pts) tile the focal plane; the ~16 field-edge CCDs fall
# back to the whole-focal-plane field (n == len(ocs_x)); keep them separate so
# they can be drawn as a faint backdrop under the per-CCD footprints.
FB_N = ocs_x.size
fp_x, fp_y, fp_v, fb_x, fb_y, fb_v, n_fp_ccd = [], [], [], [], [], [], 0
for d in DETECTORS:
    c = calibs[d]
    x = np.asarray(c.field_x, float)
    y = np.asarray(c.field_y, float)
    v = np.asarray(c.values, float)
    if x.size >= FB_N:
        fb_x.append(x); fb_y.append(y); fb_v.append(v)
    else:
        fp_x.append(x); fp_y.append(y); fp_v.append(v); n_fp_ccd += 1
_cat = lambda L: (np.concatenate(L) if L else np.zeros(0))
_catv = lambda L: (np.concatenate(L) if L else np.zeros((0, len(noll))))
fp_x, fp_y, fp_v = _cat(fp_x), _cat(fp_y), _catv(fp_v)
fb_x, fb_y, fb_v = _cat(fb_x), _cat(fb_y), _catv(fb_v)

# a representative single detector for the per-footprint cells further down
DET = 94 if 94 in calibs else DETECTORS[len(DETECTORS) // 2]
calib = calibs[DET]
fx = np.asarray(calib.field_x, float)
fy = np.asarray(calib.field_y, float)

print(f"Noll indices : {noll}")
print(f"OCS : {ocs_x.size} pts (shared disk to data edge r={DATA_R:.3f}°)")
print(f"CCS : {fp_x.size} footprint pts across {n_fp_ccd} CCDs "
      f"+ {fb_x.size} whole-FP pts across {fb_x.size // FB_N} fallback CCDs")

## Every Zernike component: OCS (telescope) and CCS (all detectors)

For each Noll index, the **OCS** map (shared telescope field, full disk) next to the
**CCS** field assembled from **all detectors**: the per-CCD footprints tile the focal
plane, with the ~16 field-edge fallback CCDs drawn as a faint backdrop. Both panels
share a colour scale. Set `SHOW` to a list to restrict which Noll indices are drawn.

In [ ]:

def _plot_map(ax, x, y, v, title, vlim, R=None, cmap="RdBu_r", s=6, alpha=1.0):
    R = DATA_R if R is None else R           # frame = data radius -> disk fills it
    fin = np.isfinite(v)
    sc = ax.scatter(x[fin], y[fin], c=v[fin], s=s, cmap=cmap, vmin=-vlim, vmax=vlim,
                    marker="o", linewidths=0.0)
    ax.add_patch(plt.Circle((0, 0), R, fill=False, ec="k", lw=0.1, alpha=0.4))
    ax.set_aspect("equal"); ax.set_xlim(-R, R); ax.set_ylim(-R, R)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("x [deg]"); ax.set_ylabel("y [deg]")
    return sc

def _vlim(*arrs, pct=98.0):
    vv = np.concatenate([np.asarray(a, float)[np.isfinite(a)] for a in arrs if np.size(a)])
    return max(float(np.percentile(np.abs(vv), pct)), 1e-6) if vv.size else 1.0

# subset of Noll indices to show (None = all)
SHOW = None
js = noll if SHOW is None else [j for j in noll if j in SHOW]
for j in js:
    k = noll.index(j)
    O = ocs_v[:, k]
    Cfp = fp_v[:, k]
    Cfb = fb_v[:, k] if fb_v.size else np.zeros(0)
    vlim = _vlim(O, Cfp, Cfb)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
    sc = _plot_map(axs[0], ocs_x, ocs_y, O, f"Z{j}  OCS (telescope, full disk)", vlim)
    if Cfb.size and j!=4 :                                   # fallback whole-FP field as backdrop
        _plot_map(axs[1], fb_x, fb_y, Cfb, "", vlim, s=3, alpha=0.35)
    _plot_map(axs[1], fp_x, fp_y, Cfp, f"Z{j}  CCS (all detectors)", vlim, s=6)
    fig.colorbar(sc, ax=axs, shrink=0.85, label="µm")
    fig.suptitle(f"IntrinsicZernikes v2 — Z{j}  ({FILT})", fontsize=12)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path


def _plot_map(ax, x, y, v, title, vlim, R=None, cmap="RdBu_r", s=6, alpha=1.0):
    # If R is not provided, use DATA_R if it exists, otherwise infer from data
    if R is None:
        try:
            R = DATA_R
        except NameError:
            R = np.nanmax(np.sqrt(np.asarray(x)**2 + np.asarray(y)**2))

    fin = np.isfinite(v)

    sc = ax.scatter(
        np.asarray(x)[fin],
        np.asarray(y)[fin],
        c=np.asarray(v)[fin],
        s=s,
        cmap=cmap,
        vmin=-vlim,
        vmax=vlim,
        marker="o",
        linewidths=0.0,
        alpha=alpha,
    )

    ax.add_patch(
        plt.Circle((0, 0), R, fill=False, ec="k", lw=0.3, alpha=0.4)
    )

    ax.set_aspect("equal")
    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")

    return sc


def _vlim(*arrs, pct=98.0):
    vals = []

    for a in arrs:
        a = np.asarray(a, dtype=float)
        if a.size:
            vals.append(a[np.isfinite(a)])

    if not vals:
        return 1.0

    vv = np.concatenate(vals)

    if vv.size == 0:
        return 1.0

    return max(float(np.percentile(np.abs(vv), pct)), 1e-6)


def make_zernike_maps_pdf(
    output_pdf="intrinsic_zernikes_maps.pdf",
    SHOW=None,
    pct=98.0,
):
    """
    Save one page per Zernike mode into a multipage PDF.

    Required existing variables:
        noll
        ocs_v, fp_v, fb_v
        ocs_x, ocs_y
        fp_x, fp_y
        fb_x, fb_y
        FILT

    Optional:
        DATA_R
    """

    output_pdf = Path(output_pdf)

    # subset of Noll indices to show
    js = list(noll) if SHOW is None else [j for j in noll if int(j) in SHOW]

    with PdfPages(output_pdf) as pdf:
        for j in js:
            j_int = int(j)
            k = list(noll).index(j)

            O = ocs_v[:, k]
            Cfp = fp_v[:, k]
            Cfb = fb_v[:, k] if fb_v.size else np.zeros(0)

            vlim = _vlim(O, Cfp, Cfb, pct=pct)

            fig, axs = plt.subplots(
                1,
                2,
                figsize=(11, 4.6),
                layout="constrained",
            )

            sc = _plot_map(
                axs[0],
                ocs_x,
                ocs_y,
                O,
                f"Z{j_int}  OCS (telescope, full disk)",
                vlim,
            )

            # Fallback whole-FP field as backdrop
            if Cfb.size and j_int != 4:
                _plot_map(
                    axs[1],
                    fb_x,
                    fb_y,
                    Cfb,
                    "",
                    vlim,
                    s=3,
                    alpha=0.35,
                )

            _plot_map(
                axs[1],
                fp_x,
                fp_y,
                Cfp,
                f"Z{j_int}  CCS (all detectors)",
                vlim,
                s=6,
            )

            fig.colorbar(sc, ax=axs, shrink=0.85, label="µm")

            fig.suptitle(
                f"IntrinsicZernikes v2 — Z{j_int}  ({FILT})",
                fontsize=12,
            )

            pdf.savefig(fig, dpi=180)
            plt.close(fig)

    print(f"Saved PDF: {output_pdf.resolve()}")

In [ ]:
make_zernike_maps_pdf("/home/g/gmegias/aos/ts_intrinsic_wavefront/notebooks/intrinsic_zernikes_v2_all_maps.pdf")

## Combined intrinsic over the detector footprint

`getIntrinsicZernikes` sums the interpolated CCS and (rotator-rotated) OCS
contributions — the aberration a consumer actually applies. Because the v3 CCS is
defined only over **this detector's footprint**, we grid over that footprint (not
the whole focal plane, which would be NaN off the CCD).

In [ ]:
ROT_TEL_POS = 0.0  # deg; rotate the OCS contribution before summing

# grid over this detector's footprint (a small margin inside its sample hull),
# since the v3 CCS is only defined there
pad = 0.02
x0, x1 = fx.min() + pad, fx.max() - pad
y0, y1 = fy.min() + pad, fy.max() - pad
n = 60
gx, gy = np.meshgrid(np.linspace(x0, x1, n), np.linspace(y0, y1, n))
px, py = gx.ravel(), gy.ravel()
z = np.asarray(calib.getIntrinsicZernikes(px, py, rotTelPos=ROT_TEL_POS), float)  # (n_pts, n_noll)

COMBINED_SHOW = [4, 5, 6, 7, 8, 11]  # edit; must be in `noll`
cols = 3
rows = int(np.ceil(len(COMBINED_SHOW) / cols))
fig, axs = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.0 * rows), layout="constrained")
axs = np.atleast_1d(axs).ravel()
for ax, j in zip(axs, COMBINED_SHOW):
    k = noll.index(j)
    m = np.isfinite(z[:, k])
    vlim = _vlim(z[:, k])
    sc = ax.scatter(px[m], py[m], c=z[m, k], s=8, cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    ax.set_aspect("equal"); ax.set_title(f"Z{j}  combined", fontsize=9)
    ax.set_xlabel("x [deg]"); ax.set_ylabel("y [deg]")
    fig.colorbar(sc, ax=ax, shrink=0.8, label="µm")
for ax in axs[len(COMBINED_SHOW):]:
    ax.axis("off")
fig.suptitle(f"IntrinsicZernikes v3 — combined  det{DET} {FILT}  rotTelPos={ROT_TEL_POS}° "
             f"(over footprint)", fontsize=12)
plt.show()

## Per-detector mean CCS Z4 (camera field + height)

Mean CCS Z4 of each detector's calib at the chosen filter — the between-CCD
variation (dominated by the per-CCD height offset). In v3 this is the mean over the
detector's footprint samples; the intra-CCD spread around it (the next cell) is the
height-map structure v3 newly captures.

In [ ]:
k4 = noll.index(4)
piston = {d: float(np.nanmean(np.asarray(calibs[d].values, float)[:, k4])) for d in DETECTORS}

ids = sorted(piston)
fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
ax.bar(ids, [piston[i] for i in ids], width=1.0, color="indianred")
ax.set_xlabel("detector id"); ax.set_ylabel("mean CCS Z4 [µm]")
ax.set_title(f"Per-detector mean CCS Z4 (height + camera field), filter {FILT}", fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.show()
print(f"Z4 spread: min={min(piston.values()):+.3f}  max={max(piston.values()):+.3f}  "
      f"ptp={np.ptp(list(piston.values())):.3f} µm")

## Intra-CCD Z4 height structure (the v3 refinement)

The whole focal plane assembled from every detector's footprint samples, coloured by
CCS Z4 — you can see the per-CCD height texture. The histogram is the intra-CCD Z4
peak-to-peak per footprint detector: the height-map structure v3 preserves that v2
(one constant per CCD) discarded.

In [ ]:
# intra-CCD Z4 structure, reusing the assembled footprint arrays (fp_*) from above
k4 = noll.index(4)
zc = fp_v[:, k4]
intra_ptp = [np.ptp(np.asarray(calibs[d].values, float)[:, k4])
             for d in DETECTORS if np.asarray(calibs[d].field_x).size < FB_N]

fig, axs = plt.subplots(1, 2, figsize=(13, 5.5), layout="constrained")
sc = axs[0].scatter(fp_x, fp_y, c=zc, s=3, cmap="viridis")
axs[0].add_patch(plt.Circle((0, 0), DATA_R, fill=False, ec="k", lw=0.6, alpha=0.4))
axs[0].set_aspect("equal"); axs[0].set_xlim(-DATA_R, DATA_R); axs[0].set_ylim(-DATA_R, DATA_R)
axs[0].set_xlabel("x [deg]"); axs[0].set_ylabel("y [deg]")
axs[0].set_title(f"CCS Z4 (µm) — per-footprint sampling, all footprint CCDs ({FILT})", fontsize=10)
fig.colorbar(sc, ax=axs[0], shrink=0.8, label="Z4 [µm]")

axs[1].hist(intra_ptp, bins=40, color="indianred")
axs[1].set_xlabel("intra-CCD Z4 peak-to-peak [µm]"); axs[1].set_ylabel("detectors")
axs[1].set_title(f"Intra-CCD Z4 structure (median {np.median(intra_ptp):.3f} µm, "
                 f"{len(intra_ptp)} footprint dets)", fontsize=10)
axs[1].grid(axis="y", alpha=0.3)
plt.show()